# P3-1 CSV별 간단 EDA 및 기술통계

대상: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1.ipynb`

구성:
- 공통 EDA 함수 셀 1개
- CSV 6개 각각 1개 코드 셀
- 각 셀은 로드, row/column 수, raw dtype, 추정 est_type, 결측/0, 후보 target, 수치/범주 기술통계, 샘플을 출력
- 상세 산출물은 `p3_1_eda_outputs/`에 CSV로 저장

In [1]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path('/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3')
OUTPUT_DIR = NOTEBOOK_DIR / 'p3_1_eda_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    '01_main_admission_grade_employment': 'P2_G2_메인_입결_A_취업진학.CSV',
    '02_salary_quartile_reference': 'P2_G2_임금분류_학부대학원_사분위기준.CSV',
    '03_salary_column_contract': 'P2_G2_임금분류_학부대학원_컬럼설명.CSV',
    '04_salary_degree_clean': 'P2_G2_임금분류_학부대학원.CSV',
    '05_regular_admission_outcome': 'P2_G2_정시입결.csv',
    '06_job_certificate_mapping': 'P2_G2_직무별_자격증매핑.CSV',
}

TARGET_PATTERNS = [
    ('입결_프록시', 'primary_target', '정시 입결 proxy. 입시 난이도/선발 결과를 나타내는 핵심 후보 y'),
    ('A비율', 'target_or_feature', '학점 A 비율. 학업성과/평가 분포 후보 y 또는 설명 변수'),
    ('CD비율', 'target_or_feature', '학점 C/D 비율. 낮은 성취 분포 후보 y 또는 설명 변수'),
    ('F비율', 'target_or_feature', '학점 F 비율. 학업 위험도 후보 y 또는 설명 변수'),
    ('전체취업률', 'target', '취업 성과 후보 y'),
    ('건보가입취업률', 'target', '건강보험 가입 기준 취업 성과 후보 y'),
    ('전체진학률', 'target', '진학 성과 후보 y'),
    ('대학원진학률', 'target', '대학원 진학 성과 후보 y'),
    ('평균소득만원', 'primary_target', '월 평균 초임소득. 임금분류 테이블의 핵심 후보 y'),
    ('중위소득만원', 'primary_target', '월 중위 초임소득. 왜도에 덜 민감한 후보 y'),
    ('평균소득_중위소득_차이만원', 'target_diagnostic', '평균과 중위의 차이. 소득 분포 왜도 진단'),
    ('평균소득_중위소득_비율', 'target_diagnostic', '평균/중위 비율. 상위 소득 영향 진단'),
    ('초임급여_', 'target_distribution', '초임급여 구간별 규모/비율. target 분포 진단'),
    ('cert_자격증취득률_pct', 'feature_or_target', '자격증 취득률. 자격증 강도 후보 변수'),
    ('cert_자격증취득수_건', 'feature_or_target', '자격증 취득 건수. 자격증 강도 후보 변수'),
    ('cert_분석대상자당자격증취득수', 'feature_or_target', '전체 분석대상자 기준 자격증 강도'),
    ('job_cert_직무비율_pct', 'target_distribution', '직무별 자격증 취득 구성비'),
    ('job_cert_대학원비율_pct', 'feature_or_target', '직무별 자격증 중 대학원 비중'),
    ('job_cert_학부대학원차이_pctp', 'diagnostic', '직무별 학부-대학원 비율 차이'),
    ('지원자_', 'target_candidate_count', '입시 수요/경쟁 규모 후보'),
    ('입학자_', 'target_candidate_count', '입학 성과/규모 후보'),
    ('재학생_', 'target_candidate_count', '재학생 규모 후보'),
    ('재적생_', 'target_candidate_count', '재적 규모 후보'),
    ('졸업자_', 'target_candidate_count', '졸업 성과/규모 후보'),
    ('취업률', 'target', '취업률 계열 후보'),
    ('진학률', 'target', '진학률 계열 후보'),
]

ID_HINTS = ('코드', '학교명', '학과명', '학과_전공', '학과_계열', '직무분류', 'column', 'source_file')
COUNT_HINTS = ('수', '건', '명', '계', '인원', '정원', '지원자', '입학자', '재학생', '재적생', '졸업자', '교원')
RATE_HINTS = ('비율', '률', 'pct', 'ratio')
QUARTILE_HINTS = ('사분위', 'quartile')


def read_csv_smart(path: Path) -> pd.DataFrame:
    last_error = None
    for enc in ('utf-8-sig', 'utf-8', 'cp949'):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception as e:
            last_error = e
    raise last_error


def numeric_like(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = series.astype('string').str.replace(',', '', regex=False).str.replace('%', '', regex=False)
    return pd.to_numeric(cleaned, errors='coerce')


def estimate_column_type(name: str, series: pd.Series) -> str:
    s = series.dropna()
    n = len(series)
    nunique = series.nunique(dropna=True)
    lname = str(name).lower()
    if pd.api.types.is_bool_dtype(series):
        return 'boolean'
    if any(h in str(name) for h in QUARTILE_HINTS):
        return 'ordered_quartile_label'
    num = numeric_like(series)
    numeric_ratio = float(num.notna().mean()) if n else 0.0
    if numeric_ratio >= 0.95:
        non_na = num.dropna()
        unique_vals = set(non_na.unique()[:20])
        if unique_vals and unique_vals <= {0, 1, 0.0, 1.0}:
            return 'binary_numeric'
        if any(h in str(name) for h in RATE_HINTS):
            return 'numeric_rate_or_ratio'
        if any(h in str(name) for h in COUNT_HINTS):
            return 'numeric_count'
        if nunique <= 12:
            return 'numeric_low_cardinality'
        return 'numeric_continuous'
    if nunique <= 2:
        return 'categorical_binary'
    if any(h in str(name) for h in ID_HINTS):
        return 'identifier_or_key'
    if nunique <= max(20, int(n * 0.02)):
        return 'categorical_low_cardinality'
    return 'text_or_high_cardinality_category'


def schema_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        num = numeric_like(s)
        numeric_ok = pd.api.types.is_numeric_dtype(s) or (num.notna().mean() >= 0.95)
        zero_count = int((num == 0).sum()) if numeric_ok else pd.NA
        rows.append({
            'column': col,
            'raw_dtype': str(s.dtype),
            'est_type': estimate_column_type(col, s),
            'non_null': int(s.notna().sum()),
            'null_count': int(s.isna().sum()),
            'null_pct': round(float(s.isna().mean() * 100), 4),
            'unique_count': int(s.nunique(dropna=True)),
            'zero_count_if_numeric': zero_count,
            'min_if_numeric': float(num.min()) if numeric_ok and num.notna().any() else np.nan,
            'max_if_numeric': float(num.max()) if numeric_ok and num.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def target_candidates(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    rows = []
    schema = schema_summary(df).set_index('column')
    for col in df.columns:
        reasons = []
        roles = []
        for pattern, role, reason in TARGET_PATTERNS:
            if pattern in col:
                reasons.append(reason)
                roles.append(role)
        if col.endswith('_사분위'):
            reasons.append('사분위 라벨. 원 수치 타겟/피처의 구간화 결과이며 순서형 범주로 사용 가능')
            roles.append('derived_ordered_category')
        if filename.endswith('컬럼설명.CSV') and col in {'role', 'meaning', 'quartile_column'}:
            reasons.append('컬럼 계약/정의 테이블의 핵심 설명 컬럼. 모델 y가 아니라 메타데이터 확인 대상')
            roles.append('metadata_key')
        if filename.endswith('사분위기준.CSV') and col in {'q1_25pct', 'q2_median', 'q3_75pct', 'min', 'max'}:
            reasons.append('사분위 기준값. 모델 y가 아니라 기준표의 수치 요약 대상')
            roles.append('reference_stat')
        if reasons:
            row = schema.loc[col].to_dict()
            row.update({
                'column': col,
                'role_guess': ' | '.join(dict.fromkeys(roles)),
                'reason': ' / '.join(dict.fromkeys(reasons)),
            })
            rows.append(row)
    if not rows:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        for col in numeric_cols[:10]:
            row = schema.loc[col].to_dict()
            row.update({'column': col, 'role_guess': 'numeric_candidate', 'reason': '명시적 target 키워드는 없지만 수치형 분석 후보'})
            rows.append(row)
    return pd.DataFrame(rows)


def numeric_describe(df: pd.DataFrame) -> pd.DataFrame:
    numeric = df.select_dtypes(include='number')
    if numeric.empty:
        return pd.DataFrame()
    desc = numeric.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc.insert(0, 'missing', numeric.isna().sum())
    desc.insert(1, 'zero_count', (numeric == 0).sum())
    return desc.reset_index(names='column')


def categorical_describe(df: pd.DataFrame, max_cols=80) -> pd.DataFrame:
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    rows = []
    for col in cat_cols[:max_cols]:
        vc = df[col].astype('string').value_counts(dropna=False).head(5)
        rows.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'unique_count': int(df[col].nunique(dropna=True)),
            'null_count': int(df[col].isna().sum()),
            'top_values': ' | '.join(f'{idx}: {val}' for idx, val in vc.items()),
        })
    return pd.DataFrame(rows)


def top_missing(schema: pd.DataFrame, n=15) -> pd.DataFrame:
    return schema.sort_values(['null_count', 'null_pct'], ascending=False).head(n)


def safe_stem(filename: str) -> str:
    stem = Path(filename).stem
    return re.sub(r'[^0-9A-Za-z가-힣_]+', '_', stem)


def run_file_eda(label: str, filename: str, sample_n=5) -> pd.DataFrame:
    path = NOTEBOOK_DIR / filename
    df = read_csv_smart(path)
    schema = schema_summary(df)
    targets = target_candidates(df, filename)
    num_desc = numeric_describe(df)
    cat_desc = categorical_describe(df)
    missing = top_missing(schema)
    sample = df.head(sample_n)

    stem = safe_stem(filename)
    schema.to_csv(OUTPUT_DIR / f'{stem}__schema_summary.csv', index=False, encoding='utf-8-sig')
    targets.to_csv(OUTPUT_DIR / f'{stem}__target_candidates.csv', index=False, encoding='utf-8-sig')
    num_desc.to_csv(OUTPUT_DIR / f'{stem}__numeric_describe.csv', index=False, encoding='utf-8-sig')
    cat_desc.to_csv(OUTPUT_DIR / f'{stem}__categorical_describe.csv', index=False, encoding='utf-8-sig')
    missing.to_csv(OUTPUT_DIR / f'{stem}__missing_top.csv', index=False, encoding='utf-8-sig')
    sample.to_csv(OUTPUT_DIR / f'{stem}__sample_head.csv', index=False, encoding='utf-8-sig')

    display(Markdown(f'## {label}: `{filename}`'))
    print(f'path: {path}')
    print(f'shape: rows={df.shape[0]:,}, columns={df.shape[1]:,}')
    print(f'total_null_cells: {int(df.isna().sum().sum()):,}')
    print(f'duplicated_rows: {int(df.duplicated().sum()):,}')
    print(f'raw_dtype_counts: {df.dtypes.astype(str).value_counts().to_dict()}')
    print(f'est_type_counts: {schema["est_type"].value_counts().to_dict()}')
    print(f'saved_prefix: {OUTPUT_DIR / stem}')

    display(Markdown('### 1) 컬럼 타입/dtype/추정 est_type 요약'))
    display(schema.head(30))
    display(Markdown('### 2) 후보 타겟/핵심 컬럼'))
    display(targets.head(30))
    display(Markdown('### 3) 결측 상위 컬럼'))
    display(missing)
    display(Markdown('### 4) 수치형 기술통계'))
    display(num_desc.head(30))
    display(Markdown('### 5) 범주형 기술통계/top values'))
    display(cat_desc.head(30))
    display(Markdown('### 6) 샘플'))
    display(sample)
    return df

print(f'NOTEBOOK_DIR={NOTEBOOK_DIR}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print('datasets=', DATASETS)


NOTEBOOK_DIR=/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3
OUTPUT_DIR=/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs
datasets= {'01_main_admission_grade_employment': 'P2_G2_메인_입결_A_취업진학.CSV', '02_salary_quartile_reference': 'P2_G2_임금분류_학부대학원_사분위기준.CSV', '03_salary_column_contract': 'P2_G2_임금분류_학부대학원_컬럼설명.CSV', '04_salary_degree_clean': 'P2_G2_임금분류_학부대학원.CSV', '05_regular_admission_outcome': 'P2_G2_정시입결.csv', '06_job_certificate_mapping': 'P2_G2_직무별_자격증매핑.CSV'}


In [2]:
df_main = run_file_eda('01 메인 입결/A/취업진학', 'P2_G2_메인_입결_A_취업진학.CSV')

## 01 메인 입결/A/취업진학: `P2_G2_메인_입결_A_취업진학.CSV`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_메인_입결_A_취업진학.CSV
shape: rows=15,727, columns=120
total_null_cells: 8,260
duplicated_rows: 0
raw_dtype_counts: {'int64': 107, 'str': 13}
est_type_counts: {'numeric_count': 91, 'numeric_continuous': 14, 'categorical_low_cardinality': 9, 'identifier_or_key': 3, 'numeric_low_cardinality': 1, 'categorical_binary': 1, 'binary_numeric': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_메인_입결_A_취업진학


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,연도,int64,numeric_low_cardinality,15727,0,0.0000,1,0,2024.0,2024.0
1,학제,str,categorical_low_cardinality,15727,0,0.0000,10,<NA>,NaN,NaN
2,대학원구분,str,categorical_binary,7467,8260,52.5211,2,<NA>,NaN,NaN
3,학교명,str,identifier_or_key,15727,0,0.0000,587,<NA>,NaN,NaN
4,본분교,str,categorical_low_cardinality,15727,0,0.0000,5,<NA>,NaN,NaN
5,시도,str,categorical_low_cardinality,15727,0,0.0000,17,<NA>,NaN,NaN
6,시군구,str,categorical_low_cardinality,15727,0,0.0000,142,<NA>,NaN,NaN
7,설립,str,categorical_low_cardinality,15727,0,0.0000,7,<NA>,NaN,NaN
8,학위과정,str,categorical_low_cardinality,15727,0,0.0000,3,<NA>,NaN,NaN
9,대계열,str,categorical_low_cardinality,15727,0,0.0000,7,<NA>,NaN,NaN


### 2) 후보 타겟/핵심 컬럼

...

,column,missing,zero_count,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,연도,0,0,15727.0,2024.000000,0.000000,2024.0,2024.0,2024.0,2024.0,2024.0,2024.0,2024.0,2024.00,2024.0
1,학과수_전체,0,2138,15727.0,1.158581,0.639161,0.0,0.0,0.0,1.0,1.0,2.0,2.0,2.00,3.0
2,학과수_석사,0,9785,15727.0,0.379093,0.487791,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.00,2.0
3,학과수_박사,0,10596,15727.0,0.326572,0.469652,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.00,2.0
4,입학정원_학부_계,0,12099,15727.0,10.951803,30.432540,0.0,0.0,0.0,0.0,0.0,0.0,57.0,131.00,729.0
5,정원내_입학정원_학부,0,12167,15727.0,10.844598,30.380549,0.0,0.0,0.0,0.0,0.0,0.0,56.0,130.74,729.0
6,정원외_입학정원_학부,0,15642,15727.0,0.107204,1.767254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,63.0
7,모집인원_학부_계,0,11819,15727.0,12.602276,33.632233,0.0,0.0,0.0,0.0,0.0,0.0,63.0,143.00,811.0
8,정원내_모집인원_학부,0,12057,15727.0,10.849431,29.539646,0.0,0.0,0.0,0.0,0.0,0.0,55.0,124.00,729.0
9,정원외_모집정원_학부,0,12886,15727.0,1.752845,9.483254,0.0,0.0,0.0,0.0,0.0,0.0,7.0,33.00,438.0


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,학제,str,10,0,대학교: 7314 | 일반대학원: 6711 | 전문대학원: 637 | 전문대학: 6...
1,대학원구분,str,2,8260,<NA>: 8260 | 부설대학원: 7235 | 대학원대학: 232
2,학교명,str,587,0,강원대학교: 356 | 건국대학교: 207 | 경북대학교: 202 | 영산대학교: ...
3,본분교,str,5,0,본교(제1캠퍼스): 13696 | 본교(제2캠퍼스): 1462 | 분교(제1캠퍼스)...
4,시도,str,17,0,서울: 3376 | 경기: 1929 | 부산: 1529 | 대전: 1181 | 강원...
5,시군구,str,142,0,서울 서대문구: 564 | 서울 성북구: 543 | 강원 춘천시: 501 | 대전 ...
6,설립,str,7,0,사립: 10351 | 국립: 4493 | 국립대법인: 355 | 특별법법인: 192...
7,학위과정,str,3,0,대학과정: 7494 | 대학원과정: 7467 | 전문대학과정: 766
8,대계열,str,7,0,공학계열: 4212 | 사회계열: 3086 | 자연계열: 2465 | 인문계열: 2...
9,중계열,str,36,0,경영ㆍ경제: 1558 | 사회과학: 1344 | 인문과학: 1270 | 생물ㆍ화학ㆍ...


### 6) 샘플

,연도,학제,대학원구분,학교명,본분교,시도,시군구,설립,학위과정,대계열,...,졸업자_박사_계,졸업자_박사_남,졸업자_박사_여,전임교원_계,전임교원_남,전임교원_여,비전임교원_계,비전임교원_남,비전임교원_여,대학원구분_clean
0,2024,대학교,NaN,국립강릉원주대학교,본교(제1캠퍼스),강원,강원 강릉시,국립,대학과정,인문계열,...,0,0,0,5,2,3,15,10,5,0
1,2024,대학교,NaN,국립강릉원주대학교,본교(제1캠퍼스),강원,강원 강릉시,국립,대학과정,인문계열,...,0,0,0,5,4,1,1,0,1,0
2,2024,대학교,NaN,국립강릉원주대학교,본교(제1캠퍼스),강원,강원 강릉시,국립,대학과정,인문계열,...,0,0,0,5,3,2,6,5,1,0
3,2024,대학교,NaN,국립강릉원주대학교,본교(제1캠퍼스),강원,강원 강릉시,국립,대학과정,인문계열,...,0,0,0,1,1,0,5,4,1,0
4,2024,대학교,NaN,국립강릉원주대학교,본교(제1캠퍼스),강원,강원 강릉시,국립,대학과정,인문계열,...,0,0,0,4,4,0,7,6,1,0


In [3]:
df_salary_quartile_ref = run_file_eda('02 임금분류 학부대학원 사분위 기준', 'P2_G2_임금분류_학부대학원_사분위기준.CSV')

## 02 임금분류 학부대학원 사분위 기준: `P2_G2_임금분류_학부대학원_사분위기준.CSV`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_임금분류_학부대학원_사분위기준.CSV
shape: rows=23, columns=10
total_null_cells: 0
duplicated_rows: 0
raw_dtype_counts: {'str': 5, 'float64': 5}
est_type_counts: {'numeric_continuous': 3, 'ordered_quartile_label': 2, 'categorical_low_cardinality': 2, 'numeric_rate_or_ratio': 2, 'identifier_or_key': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_임금분류_학부대학원_사분위기준


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,column,str,identifier_or_key,23,0,0.0,23,<NA>,NaN,NaN
1,quartile_column,str,ordered_quartile_label,23,0,0.0,23,<NA>,NaN,NaN
2,source_definition,str,categorical_low_cardinality,23,0,0.0,12,<NA>,NaN,NaN
3,q1_25pct,float64,numeric_rate_or_ratio,23,0,0.0,23,0,0.724943,7032.75
4,q2_median,float64,numeric_continuous,23,0,0.0,23,0,0.891492,12685.00
5,q3_75pct,float64,numeric_rate_or_ratio,23,0,0.0,23,0,1.194899,20383.25
6,min,float64,numeric_continuous,23,0,0.0,23,0,0.413953,1752.00
7,max,float64,numeric_continuous,23,0,0.0,23,0,1.441111,103757.00
8,quartile_scope,str,ordered_quartile_label,23,0,0.0,1,<NA>,NaN,NaN
9,interpretation,str,categorical_low_cardinality,23,0,0.0,7,<NA>,NaN,NaN


### 2) 후보 타겟/핵심 컬럼

...

,column,missing,zero_count,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,q1_25pct,0,0,23.0,893.517443,1736.355252,0.724943,0.811320,1.127563,9.483200,40.475000,954.625,3810.000,6333.535,7032.75
1,q2_median,0,0,23.0,1890.406578,3494.023615,0.891492,0.951194,1.228514,14.337522,51.850000,1916.750,8669.650,11827.110,12685.00
2,q3_75pct,0,0,23.0,3242.046809,5839.621632,1.194899,1.200179,1.314510,18.424849,67.125000,3139.000,14340.525,19078.155,20383.25
3,min,0,0,23.0,293.931044,487.432321,0.413953,0.438849,0.577497,3.651837,25.100000,315.350,1333.600,1669.940,1752.00
4,max,0,0,23.0,11502.596924,24206.421745,1.441111,1.457681,1.614784,28.777915,85.695718,12206.000,43509.400,90675.360,103757.00


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,column,str,23,0,평균소득만원: 1 | 중위소득만원: 1 | 평균소득_중위소득_차이만원: 1 | 평균...
1,quartile_column,str,23,0,평균소득만원_사분위: 1 | 중위소득만원_사분위: 1 | 평균소득_중위소득_차이만원...
2,source_definition,str,12,0,초임급여 원자료 기반 파생 컬럼: 12 | 원자료 '월 평균 소득액 - 평균소득'....
3,quartile_scope,str,1,0,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준: 23
4,interpretation,str,7,0,Q4는 해당 값이 큰 집단: 5 | Q4는 해당 급여구간 인원 규모가 큰 집단: 5...


### 6) 샘플

,column,quartile_column,source_definition,q1_25pct,q2_median,q3_75pct,min,max,quartile_scope,interpretation
0,평균소득만원,평균소득만원_사분위,"원자료 '월 평균 소득액 - 평균소득'. 분석대상자의 월 초임 소득 평균, 단위 만원",301.200000,337.800000,433.2000,234.700000,648.500000,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준,Q4는 해당 값이 큰 집단
1,중위소득만원,중위소득만원_사분위,"원자료 '월 평균 소득액 - 중위소득'. 분석대상자의 월 초임 소득 중앙값, 단위 만원",250.125000,286.200000,407.1500,217.500000,500.000000,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준,Q4는 해당 값이 큰 집단
2,평균소득_중위소득_차이만원,평균소득_중위소득_차이만원_사분위,평균소득만원 - 중위소득만원. 평균이 중위보다 얼마나 높은지 보는 분포 왜도 진단값...,32.150000,51.850000,67.1250,13.100000,198.500000,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준,Q4는 해당 값이 큰 집단
3,평균소득_중위소득_비율,평균소득_중위소득_비율_사분위,평균소득만원 / 중위소득만원. 1보다 클수록 상위 소득이 평균을 끌어올린 정도가 큼,1.117565,1.162866,1.2189,1.030947,1.441111,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준,Q4는 해당 값이 큰 집단
4,초임급여_100만원미만_명,초임급여_100만원미만_명_사분위,초임급여 원자료 기반 파생 컬럼,136.500000,295.500000,531.5000,70.000000,1490.000000,P2_G2_임금분류_학부대학원.CSV 전체 14행 기준,Q4는 해당 급여구간 인원 규모가 큰 집단


In [4]:
df_salary_column_contract = run_file_eda('03 임금분류 학부대학원 컬럼설명', 'P2_G2_임금분류_학부대학원_컬럼설명.CSV')

## 03 임금분류 학부대학원 컬럼설명: `P2_G2_임금분류_학부대학원_컬럼설명.CSV`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_임금분류_학부대학원_컬럼설명.CSV
shape: rows=66, columns=10
total_null_cells: 43
duplicated_rows: 0
raw_dtype_counts: {'str': 10}
est_type_counts: {'categorical_low_cardinality': 4, 'identifier_or_key': 3, 'text_or_high_cardinality_category': 1, 'ordered_quartile_label': 1, 'categorical_binary': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_임금분류_학부대학원_컬럼설명


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,column,str,identifier_or_key,66,0,0.0000,66,<NA>,NaN,NaN
1,role,str,categorical_low_cardinality,66,0,0.0000,9,<NA>,NaN,NaN
2,semantic_type,str,categorical_low_cardinality,66,0,0.0000,5,<NA>,NaN,NaN
3,source_file,str,identifier_or_key,66,0,0.0000,5,<NA>,NaN,NaN
4,source_raw_column,str,identifier_or_key,66,0,0.0000,65,<NA>,NaN,NaN
5,unit,str,categorical_low_cardinality,66,0,0.0000,10,<NA>,NaN,NaN
6,meaning,str,text_or_high_cardinality_category,66,0,0.0000,50,<NA>,NaN,NaN
7,quartile_column,str,ordered_quartile_label,23,43,65.1515,23,<NA>,NaN,NaN
8,zero_policy,str,categorical_low_cardinality,66,0,0.0000,3,<NA>,NaN,NaN
9,null_policy,str,categorical_binary,66,0,0.0000,2,<NA>,NaN,NaN


### 2) 후보 타겟/핵심 컬럼

...

""


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,column,str,66,0,기준연도: 1 | 대학원여부: 1 | 계열: 1 | 분석대상자_명: 1 | 평균소득...
1,role,str,9,0,quartile_label: 23 | feature: 23 | target_dist...
2,semantic_type,str,5,0,numeric: 40 | ordered_category: 23 | year: 1 |...
3,source_file,str,5,0,P2_G2_임금분류_학부대학원.CSV 내부 계산: 23 | 2024_계열별 초임급여...
4,source_raw_column,str,65,0,"월 평균 소득액, 월 평균 소득액.1: 2 | 기준연도: 1 | 구분: 1 | 계열..."
5,unit,str,10,0,Q1~Q4: 23 | %: 18 | 명: 15 | 만원: 3 | 건/명: 2
6,meaning,str,50,0,기업유형별 취업 구성값. 기업유형은 서로 배타적인 경쟁형 구성: 16 | 행을 식별...
7,quartile_column,str,23,43,<NA>: 43 | 평균소득만원_사분위: 1 | 중위소득만원_사분위: 1 | 평균소...
8,zero_policy,str,3,0,관측된 0은 실제 0으로 유지: 42 | 해당 없음: 23 | 0은 학부를 의미하는...
9,null_policy,str,2,0,최종 CSV에 NULL이 남으면 실질 결측으로 검토: 65 | NULL이면 학부/대...


### 6) 샘플

,column,role,semantic_type,source_file,source_raw_column,unit,meaning,quartile_column,zero_policy,null_policy
0,기준연도,key,year,원자료 공통 키,기준연도,year,행을 식별하는 기준 키,NaN,관측된 0은 실제 0으로 유지,최종 CSV에 NULL이 남으면 실질 결측으로 검토
1,대학원여부,binary_key,binary,원자료 공통 키,구분,0/1,"기존 구분 컬럼에서 학부는 0, 대학원은 1로 변환한 모델 입력용 바이너리",NaN,0은 학부를 의미하는 정상 범주값,NULL이면 학부/대학원 구분 매핑 실패
2,계열,key,category,원자료 공통 키,계열,category,행을 식별하는 기준 키,NaN,관측된 0은 실제 0으로 유지,최종 CSV에 NULL이 남으면 실질 결측으로 검토
3,분석대상자_명,denominator,numeric,2024_계열별 초임급여 현황.xlsx,분석대상자,명,"조사기준일 당시 건강보험 직장가입자 중 가입된 회사명이 1개이면서 기업명, 기업규모...",NaN,관측된 0은 실제 0으로 유지,최종 CSV에 NULL이 남으면 실질 결측으로 검토
4,평균소득만원,primary_target,numeric,2024_계열별 초임급여 현황.xlsx,월 평균 소득액 - 평균소득,만원,"원자료 '월 평균 소득액 - 평균소득'. 분석대상자의 월 초임 소득 평균, 단위 만원",평균소득만원_사분위,관측된 0은 실제 0으로 유지,최종 CSV에 NULL이 남으면 실질 결측으로 검토


In [5]:
df_salary_degree = run_file_eda('04 임금분류 학부대학원 클린 테이블', 'P2_G2_임금분류_학부대학원.CSV')

## 04 임금분류 학부대학원 클린 테이블: `P2_G2_임금분류_학부대학원.CSV`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_임금분류_학부대학원.CSV
shape: rows=14, columns=66
total_null_cells: 0
duplicated_rows: 0
raw_dtype_counts: {'str': 24, 'float64': 24, 'int64': 18}
est_type_counts: {'ordered_quartile_label': 23, 'numeric_rate_or_ratio': 19, 'numeric_count': 18, 'numeric_continuous': 3, 'numeric_low_cardinality': 1, 'binary_numeric': 1, 'categorical_low_cardinality': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_임금분류_학부대학원


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,기준연도,int64,numeric_low_cardinality,14,0,0.0,1,0,2024.000000,2024.000000
1,대학원여부,int64,binary_numeric,14,0,0.0,2,7,0.000000,1.000000
2,계열,str,categorical_low_cardinality,14,0,0.0,7,<NA>,NaN,NaN
3,분석대상자_명,int64,numeric_count,14,0,0.0,14,0,2304.000000,72776.000000
4,평균소득만원,float64,numeric_continuous,14,0,0.0,14,0,234.700000,648.500000
5,평균소득만원_사분위,str,ordered_quartile_label,14,0,0.0,4,<NA>,NaN,NaN
6,중위소득만원,float64,numeric_continuous,14,0,0.0,14,0,217.500000,500.000000
7,중위소득만원_사분위,str,ordered_quartile_label,14,0,0.0,4,<NA>,NaN,NaN
8,평균소득_중위소득_차이만원,float64,numeric_continuous,14,0,0.0,14,0,13.100000,198.500000
9,평균소득_중위소득_차이만원_사분위,str,ordered_quartile_label,14,0,0.0,4,<NA>,NaN,NaN


### 2) 후보 타겟/핵심 컬럼

...

,column,missing,zero_count,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,기준연도,0,0,14.0,2024.000000,0.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000
1,대학원여부,0,7,14.0,0.500000,0.518875,0.000000,0.000000,0.000000,0.000000,0.500000,1.000000,1.000000,1.000000,1.000000
2,분석대상자_명,0,0,14.0,22803.714286,22031.418019,2304.000000,2615.350000,3860.750000,7433.250000,16207.000000,22711.250000,68111.600000,71843.120000,72776.000000
3,평균소득만원,0,0,14.0,385.442857,130.507664,234.700000,238.015000,251.275000,301.200000,337.800000,433.200000,625.100000,643.820000,648.500000
4,중위소득만원,0,0,14.0,325.492857,96.243213,217.500000,219.814000,229.070000,250.125000,286.200000,407.150000,480.695000,496.139000,500.000000
5,평균소득_중위소득_차이만원,0,0,14.0,59.950000,47.659590,13.100000,13.633000,15.765000,32.150000,51.850000,67.125000,142.600000,187.320000,198.500000
6,평균소득_중위소득_비율,0,0,14.0,1.175692,0.098130,1.030947,1.037205,1.062234,1.117565,1.162866,1.218900,1.311085,1.415106,1.441111
7,초임급여_100만원미만_명,0,0,14.0,429.714286,420.627449,70.000000,70.390000,71.950000,136.500000,295.500000,531.500000,1171.500000,1426.300000,1490.000000
8,초임급여_100~200만원미만_명,0,0,14.0,2137.000000,2328.700628,199.000000,201.730000,212.650000,317.750000,1321.000000,2746.500000,6071.450000,7281.490000,7584.000000
9,초임급여_200~300만원미만_명,0,0,14.0,10375.000000,11735.202512,800.000000,827.560000,937.800000,1672.000000,5825.500000,13346.250000,33771.250000,34933.450000,35224.000000


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,계열,str,7,0,공학: 2 | 의약: 2 | 인문: 2 | 사회: 2 | 자연: 2
1,평균소득만원_사분위,str,4,0,Q1_하위25%: 4 | Q4_상위25%: 4 | Q2_25~50%: 3 | Q3_...
2,중위소득만원_사분위,str,4,0,Q1_하위25%: 4 | Q4_상위25%: 4 | Q3_50~75%: 3 | Q2_...
3,평균소득_중위소득_차이만원_사분위,str,4,0,Q1_하위25%: 4 | Q4_상위25%: 4 | Q2_25~50%: 3 | Q3_...
4,평균소득_중위소득_비율_사분위,str,4,0,Q1_하위25%: 4 | Q4_상위25%: 4 | Q2_25~50%: 3 | Q3_...
5,초임급여_100만원미만_명_사분위,str,4,0,Q4_상위25%: 4 | Q1_하위25%: 4 | Q3_50~75%: 3 | Q2_...
6,초임급여_100~200만원미만_명_사분위,str,4,0,Q4_상위25%: 4 | Q1_하위25%: 4 | Q3_50~75%: 3 | Q2_...
7,초임급여_200~300만원미만_명_사분위,str,4,0,Q4_상위25%: 4 | Q1_하위25%: 4 | Q3_50~75%: 3 | Q2_...
8,초임급여_300~400만원미만_명_사분위,str,4,0,Q4_상위25%: 4 | Q1_하위25%: 4 | Q3_50~75%: 3 | Q2_...
9,초임급여_400만원이상_명_사분위,str,4,0,Q4_상위25%: 4 | Q1_하위25%: 4 | Q3_50~75%: 3 | Q2_...


### 6) 샘플

,기준연도,대학원여부,계열,분석대상자_명,평균소득만원,평균소득만원_사분위,중위소득만원,중위소득만원_사분위,평균소득_중위소득_차이만원,평균소득_중위소득_차이만원_사분위,...,company_국가및지방_명,company_국가및지방_pct,company_공공기관_명,company_공공기관_pct,company_비영리_명,company_비영리_pct,company_기타_명,company_기타_pct,company_대중견기업_pct,company_공공비영리_pct
0,2024,0,공학,72776,328.1,Q2_25~50%,290.0,Q3_50~75%,38.1,Q2_25~50%,...,5293,7.273002,2038,2.800374,2781,3.821315,522,0.717269,32.198802,13.894691
1,2024,0,의약,41094,314.3,Q2_25~50%,282.4,Q2_25~50%,31.9,Q1_하위25%,...,1345,3.272984,1284,3.124544,17230,41.928262,191,0.464788,2.596486,48.325790
2,2024,0,인문,18844,303.9,Q2_25~50%,250.0,Q1_하위25%,53.9,Q3_50~75%,...,3055,16.212057,459,2.435789,2487,13.197835,624,3.311399,19.289960,31.845680
3,2024,0,사회,65600,300.3,Q1_하위25%,250.5,Q2_25~50%,49.8,Q2_25~50%,...,9156,13.957317,2033,3.099085,10277,15.666159,3914,5.966463,18.635671,32.722561
4,2024,0,자연,22406,281.8,Q1_하위25%,248.9,Q1_하위25%,32.9,Q2_25~50%,...,2097,9.359100,612,2.731411,2275,10.153530,411,1.834330,21.980719,22.244042


In [6]:
df_regular_admission = run_file_eda('05 정시입결/학점/취업진학', 'P2_G2_정시입결.csv')

## 05 정시입결/학점/취업진학: `P2_G2_정시입결.csv`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_정시입결.csv
shape: rows=10,242, columns=17
total_null_cells: 27,965
duplicated_rows: 0
raw_dtype_counts: {'float64': 12, 'str': 4, 'int64': 1}
est_type_counts: {'text_or_high_cardinality_category': 6, 'identifier_or_key': 3, 'numeric_rate_or_ratio': 3, 'categorical_low_cardinality': 3, 'numeric_low_cardinality': 1, 'categorical_binary': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_정시입결


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,기준연도,int64,numeric_low_cardinality,10242,0,0.0000,1,0,2024.0,2024.000000
1,학교명,str,identifier_or_key,10242,0,0.0000,227,<NA>,NaN,NaN
2,학과_전공,str,identifier_or_key,10242,0,0.0000,4946,<NA>,NaN,NaN
3,학과_계열,str,identifier_or_key,10242,0,0.0000,4195,<NA>,NaN,NaN
4,입결_프록시,float64,text_or_high_cardinality_category,3737,6505,63.5130,1671,0,0.1,100.000000
5,A비율,float64,numeric_rate_or_ratio,10242,0,0.0000,9236,319,0.0,100.000000
6,CD비율,float64,numeric_rate_or_ratio,10242,0,0.0000,9006,475,0.0,100.000000
7,F비율,float64,numeric_rate_or_ratio,10242,0,0.0000,7459,1808,0.0,100.000000
8,전체취업률,float64,text_or_high_cardinality_category,7477,2765,26.9967,1135,78,0.0,100.000000
9,건보가입취업률,float64,text_or_high_cardinality_category,7477,2765,26.9967,1196,136,0.0,100.000000


### 2) 후보 타겟/핵심 컬럼

...

,column,missing,zero_count,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,기준연도,0,0,10242.0,2024.000000,0.000000,2024.0,2024.0000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000
1,입결_프록시,6505,0,3737.0,69.656123,19.072921,0.1,3.7734,30.764000,61.400000,72.500000,82.870000,94.170000,97.782000,100.000000
2,A비율,0,319,10242.0,41.726427,17.010999,0.0,0.0000,14.643038,32.476539,39.791552,49.940472,73.036088,92.857143,100.000000
3,CD비율,0,475,10242.0,18.606950,12.729525,0.0,0.0000,0.833552,10.412399,17.967907,24.792164,35.551012,66.666667,100.000000
4,F비율,0,1808,10242.0,3.937010,9.650919,0.0,0.0000,0.000000,0.513097,1.695090,3.510926,12.752200,50.737500,100.000000
5,전체취업률,2765,78,7477.0,61.586546,17.444647,0.0,0.0000,33.333333,50.000000,62.068966,72.000000,91.666667,100.000000,100.000000
6,건보가입취업률,2765,136,7477.0,52.675705,19.217385,0.0,0.0000,20.000000,41.176471,52.941176,64.000000,86.206897,100.000000,100.000000
7,전체진학률,2655,3098,7587.0,8.373070,13.086164,0.0,0.0000,0.000000,0.000000,2.941176,11.111111,35.897436,57.526923,100.000000
8,전문대학진학률,2655,7339,7587.0,0.109347,0.945585,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,3.448276,50.000000
9,대학진학률,2655,7077,7587.0,0.201633,0.987264,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,1.470588,4.761905,25.000000


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,학교명,str,227,0,경북대학교: 158 | 강원대학교: 154 | 신라대학교: 144 | 동의대학교: ...
1,학과_전공,str,4946,0,간호학과: 106 | 사회복지학과: 93 | 식품영양학과: 72 | 경영학과: 68...
2,학과_계열,str,4195,0,경영: 120 | 사회복지: 114 | 간호: 113 | 컴퓨터공: 83 | 식품영...
3,학점포기제유무,str,2,0,X: 9093 | O: 1149


### 6) 샘플

,기준연도,학교명,학과_전공,학과_계열,입결_프록시,A비율,CD비율,F비율,전체취업률,건보가입취업률,전체진학률,전문대학진학률,대학진학률,대학원진학률,국내진학률,국외진학률,학점포기제유무
0,2024,가야대학교(김해),간호학과,간호,4.0,35.114053,25.497462,1.185804,74.074074,74.074074,0.000000,0.0,0.0,0.000000,0.000000,0.0,X
1,2024,가야대학교(김해),경영물류학과,경영물류,NaN,34.833333,13.333333,12.166667,69.230769,61.538462,0.000000,0.0,0.0,0.000000,0.000000,0.0,X
2,2024,가야대학교(김해),경찰소방학과,경찰소방,NaN,39.507271,21.864212,1.239669,61.111111,50.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,X
3,2024,가야대학교(김해),귀금속주얼리학과,귀금속주얼리,NaN,51.785714,1.428571,5.000000,36.363636,36.363636,8.333333,0.0,0.0,8.333333,8.333333,0.0,X
4,2024,가야대학교(김해),물리치료학과,물리치료,NaN,27.349869,32.314046,5.576489,91.071429,91.071429,1.754386,0.0,0.0,1.754386,1.754386,0.0,X


In [7]:
df_job_cert_mapping = run_file_eda('06 직무별 자격증 매핑', 'P2_G2_직무별_자격증매핑.CSV')

## 06 직무별 자격증 매핑: `P2_G2_직무별_자격증매핑.CSV`

path: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/P2_G2_직무별_자격증매핑.CSV
shape: rows=24, columns=26
total_null_cells: 0
duplicated_rows: 0
raw_dtype_counts: {'str': 9, 'float64': 8, 'bool': 6, 'int64': 3}
est_type_counts: {'boolean': 6, 'numeric_count': 4, 'numeric_rate_or_ratio': 4, 'text_or_high_cardinality_category': 4, 'categorical_low_cardinality': 3, 'numeric_continuous': 3, 'identifier_or_key': 1, 'categorical_binary': 1}
saved_prefix: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/P2_G2_직무별_자격증매핑


### 1) 컬럼 타입/dtype/추정 est_type 요약

,column,raw_dtype,est_type,non_null,null_count,null_pct,unique_count,zero_count_if_numeric,min_if_numeric,max_if_numeric
0,직무분류,str,identifier_or_key,24,0,0.0,24,<NA>,NaN,NaN
1,job_cert_고등교육기관_건,float64,numeric_count,24,0,0.0,24,0,11.000000,3.034470e+05
2,job_cert_학부_건,float64,numeric_count,24,0,0.0,24,0,4.000000,2.337000e+05
3,job_cert_대학원_건,float64,numeric_count,24,0,0.0,24,0,7.000000,6.974700e+04
4,job_cert_직무비율_pct,float64,numeric_rate_or_ratio,24,0,0.0,24,0,0.003625,1.000000e+02
5,job_cert_학부비율_pct,float64,numeric_rate_or_ratio,24,0,0.0,24,0,23.560768,9.084055e+01
6,job_cert_대학원비율_pct,float64,numeric_rate_or_ratio,24,0,0.0,24,0,9.159448,7.643923e+01
7,job_cert_학부대학원차이_건,float64,numeric_count,24,0,0.0,24,0,-496.000000,1.639530e+05
8,job_cert_학부대학원차이_pctp,float64,numeric_rate_or_ratio,24,0,0.0,24,0,-52.878465,8.168110e+01
9,job_cert_대학원우세_flag,bool,boolean,24,0,0.0,2,22,0.000000,1.000000e+00


### 2) 후보 타겟/핵심 컬럼

...

,column,missing,zero_count,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,job_cert_고등교육기관_건,0,0,24.0,25287.250000,64720.989912,11.000000,23.190000,90.100000,983.000000,5215.000000,16347.000000,1.132808e+05,2.630831e+05,3.034470e+05
1,job_cert_학부_건,0,0,24.0,19475.000000,49402.800125,4.000000,11.590000,63.400000,642.750000,4137.500000,13598.750000,8.219640e+04,2.011893e+05,2.337000e+05
2,job_cert_대학원_건,0,0,24.0,5812.250000,15431.859144,7.000000,11.140000,25.300000,351.250000,841.500000,2380.250000,3.169385e+04,6.189388e+04,6.974700e+04
3,job_cert_직무비율_pct,0,0,24.0,8.333333,21.328598,0.003625,0.007642,0.029692,0.323945,1.718587,5.387102,3.733133e+01,8.669822e+01,1.000000e+02
4,job_cert_학부비율_pct,0,0,24.0,74.699434,16.826028,23.560768,26.505427,39.509732,69.945325,81.707552,84.639499,8.944661e+01,9.053126e+01,9.084055e+01
5,job_cert_대학원비율_pct,0,0,24.0,25.300566,16.826028,9.159448,9.468741,10.553385,15.360501,18.292448,30.054675,6.049027e+01,7.349457e+01,7.643923e+01
6,job_cert_학부대학원차이_건,0,0,24.0,13662.750000,34187.005403,-496.000000,-382.610000,-1.050000,236.750000,3084.000000,9494.000000,5.111200e+04,1.392954e+05,1.639530e+05
7,job_cert_학부대학원차이_pctp,0,0,24.0,49.398867,33.652056,-52.878465,-46.989145,-20.980535,39.890650,63.415104,69.278998,7.889323e+01,8.106252e+01,8.168110e+01
8,guide_match_dept_n,0,2,24.0,170.250000,163.120721,0.000000,0.000000,3.600000,65.500000,127.500000,210.000000,4.423000e+02,6.685900e+02,7.300000e+02
9,guide_match_school_sum,0,2,24.0,406.500000,390.577054,0.000000,0.000000,11.250000,112.750000,385.000000,445.000000,1.276750e+03,1.497830e+03,1.539000e+03


### 5) 범주형 기술통계/top values

,column,dtype,unique_count,null_count,top_values
0,직무분류,str,24,0,전체: 1 | 경영·회계·사무: 1 | 교육·자연과학·사회과학: 1 | 보건·의료:...
1,primary_industry_candidate,str,16,0,C.제조업: 7 | Q.보건·사회복지업: 2 | I.숙박·음식점업: 2 | 매핑대상...
2,industry_candidates,str,22,0,C.제조업: 2 | C.제조업 | R.예술·스포츠·여가업: 2 | 매핑대상아님: 1...
3,mapping_type,str,18,0,direct_domain_match: 6 | manufacturing_domain_...
4,confidence,str,5,0,medium: 10 | high: 8 | low: 4 | exclude: 1 | u...
5,guide_match_top_departments,str,23,0,"근거없음: 2 | 경영(370,552) | 경영학(59,542) | 글로벌경영(33..."
6,guide_terms,str,23,0,"근거없음: 2 | 경영, 회계, 세무, 사무, 비서: 1 | 교육, 자연과학, 사회..."
7,decision_note,str,24,0,집계 행이므로 산업분류 매핑 대상에서 제외: 1 | 직무가 산업이 아니라 기능에 가...
8,cleaning_policy,str,1,0,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음: 24


### 6) 샘플

,직무분류,job_cert_고등교육기관_건,job_cert_학부_건,job_cert_대학원_건,job_cert_직무비율_pct,job_cert_학부비율_pct,job_cert_대학원비율_pct,job_cert_학부대학원차이_건,job_cert_학부대학원차이_pctp,job_cert_대학원우세_flag,...,review_needed,join_now,mapping_missing_flag,guide_match_dept_n,guide_match_school_sum,guide_match_student_sum,guide_match_top_departments,guide_terms,decision_note,cleaning_policy
0,전체,303447.0,233700.0,69747.0,100.000000,77.015097,22.984903,163953.0,54.030193,False,...,True,False,False,0,0,0,근거없음,근거없음,집계 행이므로 산업분류 매핑 대상에서 제외,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음
1,경영·회계·사무,127952.0,92349.0,35603.0,42.166177,72.174722,27.825278,56746.0,44.349444,False,...,True,False,False,298,757,893316,"경영(370,552) | 경영학(59,542) | 글로벌경영(33,531) | 관광...","경영, 회계, 세무, 사무, 비서",직무가 산업이 아니라 기능에 가까워 단일 산업으로 확정하면 위험,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음
2,교육·자연과학·사회과학,11.0,4.0,7.0,0.003625,36.363636,63.636364,-3.0,-27.272727,True,...,True,False,False,463,1539,1236256,"유아교육과(96,382) | 물리치료(79,375) | 화학공(55,905) | 교...","교육, 자연과학, 사회과학, 수학, 물리, 화학, 생명과학, 사회학",건수가 매우 작고 교육/연구 기능이 섞여 후속 검토 필요,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음
3,보건·의료,938.0,221.0,717.0,0.309115,23.560768,76.439232,-496.0,-52.878465,True,...,False,False,False,101,437,990409,"간호(611,529) | 물리치료(79,375) | 임상병리(49,311) | 치위...","보건, 의료, 간호, 임상병리, 물리치료, 치위생, 방사선, 응급구조",학과명 가이드의 보건/의료/간호 계열과 산업 대분류가 직접 대응,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음
4,사회복지·종교,1247.0,715.0,532.0,0.410945,57.337610,42.662390,183.0,14.675221,False,...,True,False,False,128,384,479724,"사회복지(283,988) | 신(30,231) | 청소년교육복지상담(16,541) ...","사회복지, 복지, 종교, 신학",사회복지는 Q가 강하지만 종교는 S 후보가 있어 분리 검토 필요,직무별 자격증은 별도 클린 매핑표로 보관하고 임금분류 테이블에는 조인하지 않음


In [8]:

# 전체 파일 단위 요약 인덱스 저장
file_rows = []
for key, filename in DATASETS.items():
    path = NOTEBOOK_DIR / filename
    df = read_csv_smart(path)
    schema = schema_summary(df)
    targets = target_candidates(df, filename)
    file_rows.append({
        'dataset_key': key,
        'filename': filename,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'total_null_cells': int(df.isna().sum().sum()),
        'duplicated_rows': int(df.duplicated().sum()),
        'numeric_columns': int(df.select_dtypes(include='number').shape[1]),
        'non_numeric_columns': int(df.shape[1] - df.select_dtypes(include='number').shape[1]),
        'candidate_target_columns': int(len(targets)),
        'top_candidate_targets': ', '.join(targets['column'].head(8).astype(str)) if len(targets) else '',
    })
summary_index = pd.DataFrame(file_rows)
summary_index.to_csv(OUTPUT_DIR / '00_p3_1_file_level_summary.csv', index=False, encoding='utf-8-sig')
display(Markdown('## 전체 파일 단위 요약'))
display(summary_index)
print(f'file_level_summary_saved={OUTPUT_DIR / "00_p3_1_file_level_summary.csv"}')


## 전체 파일 단위 요약

,dataset_key,filename,rows,columns,total_null_cells,duplicated_rows,numeric_columns,non_numeric_columns,candidate_target_columns,top_candidate_targets
0,01_main_admission_grade_employment,P2_G2_메인_입결_A_취업진학.CSV,15727,120,8260,0,107,13,66,"지원자_전체_계, 지원자_전체_남, 지원자_전체_여, 지원자_석사_계, 지원자_석사..."
1,02_salary_quartile_reference,P2_G2_임금분류_학부대학원_사분위기준.CSV,23,10,0,0,5,5,5,"q1_25pct, q2_median, q3_75pct, min, max"
2,03_salary_column_contract,P2_G2_임금분류_학부대학원_컬럼설명.CSV,66,10,43,0,0,10,3,"role, meaning, quartile_column"
3,04_salary_degree_clean,P2_G2_임금분류_학부대학원.CSV,14,66,0,0,42,24,42,"평균소득만원, 평균소득만원_사분위, 중위소득만원, 중위소득만원_사분위, 평균소득_중..."
4,05_regular_admission_outcome,P2_G2_정시입결.csv,10242,17,27965,0,13,4,12,"입결_프록시, A비율, CD비율, F비율, 전체취업률, 건보가입취업률, 전체진학률,..."
5,06_job_certificate_mapping,P2_G2_직무별_자격증매핑.CSV,24,26,0,0,11,15,3,"job_cert_직무비율_pct, job_cert_대학원비율_pct, job_cer..."


file_level_summary_saved=/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1_eda_outputs/00_p3_1_file_level_summary.csv


# P3-1 Final Contract Build

이 섹션은 `runner/P2_G3_AGENT_PROMPT_LOCAL1.md`의 Local Agent 1 계약을 실제 산출물로 고정한다.

- 기존 EDA 셀은 보존한다.
- 원본 CSV는 수정하지 않는다.
- 출력은 `p3_1/`, `shared/`, `qa/`, `logs/` 아래에 저장한다.
- P3에서는 모델링, 결측대체, 스케일링, 원-핫 인코딩, 타깃 선택을 수행하지 않는다.


In [9]:
# Legacy P3-1 Final Contract Build is intentionally not rerun in the RED remediation pass.
# It writes to public shared/log/qa paths. The remediation section below writes only to
# p4_handoff_candidate/*, logs/local1, and qa/local1.
print("SKIPPED legacy public-write P3-1 Final Contract Build during remediation run.")

SKIPPED legacy public-write P3-1 Final Contract Build during remediation run.


In [10]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path('/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3')
manifest = json.loads((NOTEBOOK_DIR / 'logs' / 'p3_1_run_manifest.json').read_text(encoding='utf-8'))

display(Markdown('## 산출물 shape/hash'))
outputs = pd.DataFrame(manifest['outputs'])
display(outputs[['logical_name', 'exists', 'rows', 'columns', 'sha256', 'path']])

display(Markdown('## 매칭·계열·outcome·split 요약'))
display(pd.DataFrame([
    {'metric': 'match_method_distribution', 'value': manifest['match_method_distribution']},
    {'metric': 'headcount_high_confidence_match_rate', 'value': manifest['headcount_high_confidence_match_rate']},
    {'metric': 'major7_mapping_distribution', 'value': manifest['major7_mapping_distribution']},
    {'metric': 'outcome_non_null_counts', 'value': manifest['outcome_non_null_counts']},
    {'metric': 'split_summary', 'value': manifest['split_summary']},
]))

display(Markdown('## QA / failed checks'))
display(pd.read_csv(NOTEBOOK_DIR / 'qa' / 'final_qa_summary.csv', encoding='utf-8-sig'))
display(pd.read_csv(NOTEBOOK_DIR / 'qa' / 'failed_checks.csv', encoding='utf-8-sig'))

d08_path = NOTEBOOK_DIR / 'shared' / 'mart_department_model_base_2024.parquet'
if d08_path.exists():
    d08 = pd.read_parquet(d08_path)
    goms_cols = [c for c in d08.columns if c.startswith('goms_')]
    ctx_cols = [c for c in d08.columns if c.startswith('ctx24_')]
    display(Markdown('## D08 context 결합률'))
    display(pd.DataFrame({
        'block': ['major_group_7', 'ctx24_context', 'goms_context'],
        'non_null_rows': [
            int(d08['major_group_7'].notna().sum()),
            int(d08[ctx_cols].notna().any(axis=1).sum()),
            int(d08[goms_cols].notna().any(axis=1).sum()),
        ],
        'total_rows': [len(d08), len(d08), len(d08)],
    }))


## 산출물 shape/hash

,logical_name,exists,rows,columns,sha256,path
0,D01 dept_headcount_master_2024.parquet,True,15727.0,175.0,9ad156f25ad71e17835e1d2c73a15515e63b49c87e1614...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
1,D02 dept_outcomes_2024.parquet,True,10242.0,32.0,871cfacbfc1020717c0ccbea64f4fad3e6adf5dd9b091b...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
2,D03 dept_master_2024_core.parquet,True,10242.0,88.0,06892a716d5fef2b87e4b0806c6f4b398e8c0ddfa6c4f8...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
3,D04 wage_reference_by_major.parquet,True,14.0,87.0,489caf15edbefa1ed0c30fdfa98dbe31096b36c219f661...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
4,D05 job_cert_bridge.parquet,True,24.0,32.0,02dd640c73448f901b08c195091f6a6f3893cdf69af23d...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
5,D08 mart_department_model_base_2024.parquet,True,10242.0,131.0,e7021282447ef1c993dcfe552d61738a2a4f990fddb1d7...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
6,bridge_outcome_headcount.csv,True,10242.0,9.0,61efdbc561b56691904b9606a162789d4760966d84357d...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
7,bridge_department_major7.csv,True,10242.0,7.0,b6e5f6d81988538b9bdd9a065a6719b083f49d21e6d32b...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
8,dim_school_split.csv,True,200.0,6.0,85c2c851ddcfd02d5ead41dbd9424124e4ef1993347842...,/home/sieg/projects-wsl/SBS_dataScience/workbo...
9,model_sample_registry.csv,True,5.0,7.0,9971a62aa8c3dbb59a79a17cd557d18e4445a2ba9b0581...,/home/sieg/projects-wsl/SBS_dataScience/workbo...


## 매칭·계열·outcome·split 요약

,metric,value
0,match_method_distribution,"{'unmatched': 5204, 'exact_normalized': 4710, ..."
1,headcount_high_confidence_match_rate,0.459969
2,major7_mapping_distribution,"{'inherited_headcount': 4711, 'exact_dictionar..."
3,outcome_non_null_counts,"{'selectivity_proxy_pct': 3737, 'a_rate_pct': ..."
4,split_summary,"[{'split': 'test', 'rows': 1199, 'schools': 30..."


## QA / failed checks

,check_name,status,detail
0,D02 canonical spine row count,PASS,"10,242 rows"
1,D03 row count after left join,PASS,"10,242 rows"
2,D04 row count,PASS,14 rows
3,D05 join policy,PASS,join_now all False
4,D08 build,PASS,BUILT
5,split leakage,PASS,max split per school=1
6,D04 industry context,WARN,industry_top3_pct and industry_hhi source colu...


,check_name,status,detail
0,D04 industry context,WARN,industry_top3_pct and industry_hhi source colu...


## D08 context 결합률

,block,non_null_rows,total_rows
0,major_group_7,9361,10242
1,ctx24_context,9361,10242
2,goms_context,9361,10242


# P3-1 Final Remediation and P4 Handoff Candidate

RED 감사 보정 전용 섹션이다. 기존 public `shared/`, `logs/`, `qa/`는 덮어쓰지 않고 `p4_handoff_candidate/local1`, `logs/local1`, `qa/local1`에 Local 1 candidate와 lineage를 생성한다.


In [11]:
from pathlib import Path
import json
import subprocess
import sys
import time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "p3_1_p4_handoff_remediation_local1.py").exists():
    PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")

script_path = PROJECT_ROOT / "scripts" / "p3_1_p4_handoff_remediation_local1.py"
started = time.perf_counter()
completed = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
notebook_section_script_elapsed_seconds = round(time.perf_counter() - started, 3)

report_path = PROJECT_ROOT / "workbook" / "p2" / "p2_3" / "logs" / "local1" / "P4_HANDOFF_LOCAL1_REPORT.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("script_elapsed_seconds_from_notebook_cell =", notebook_section_script_elapsed_seconds)
print("script_reported_runtime_seconds =", report["script_runtime_seconds"])
print("final_status =", report["final_status"])
print("stdout_tail:")
print(completed.stdout[-3000:])


script_elapsed_seconds_from_notebook_cell = 65.275
script_reported_runtime_seconds = 64.703
final_status = READY_FOR_REAUDIT
stdout_tail:
f6610dae69a2d5e49c47e5",
        "mtime": "2026-07-12T21:42:11",
        "size_bytes": 77696
      },
      {
        "logical_name": "D05",
        "path": "/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p3_1/job_cert_bridge.parquet",
        "exists": true,
        "shape": [
          24,
          32
        ],
        "sha256": "02dd640c73448f901b08c195091f6a6f3893cdf69af23d79139ebdac8f1cc33f",
        "mtime": "2026-07-12T21:42:11",
        "size_bytes": 31747
      },
      {
        "logical_name": "existing_D08",
        "path": "/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/shared/mart_department_model_base_2024.parquet",
        "exists": true,
        "shape": [
          10242,
          131
        ],
        "sha256": "e7021282447ef1c993dcfe552d61738a2a4f990fddb1d7b85676c9d5a10cdb01",
        "mtime": "2026-07-12T2

In [12]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "workbook" / "p2" / "p2_3").exists():
    PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")

base = PROJECT_ROOT / "workbook" / "p2" / "p2_3"
report = json.loads((base / "logs" / "local1" / "P4_HANDOFF_LOCAL1_REPORT.json").read_text(encoding="utf-8"))
qa = pd.read_csv(base / "qa" / "local1" / "final_qa_summary.csv", encoding="utf-8-sig")
manifest = json.loads((base / "p4_handoff_candidate" / "local1" / "P4_HANDOFF_MANIFEST.json").read_text(encoding="utf-8"))

summary = pd.DataFrame([
    {"metric": "D01 v2 shape", "value": str(report["d01_shape"])},
    {"metric": "D01 v2 hash", "value": report["d01_hash"]},
    {"metric": "D03 v2 shape", "value": str(report["d03_shape"])},
    {"metric": "D03 v2 hash", "value": report["d03_hash"]},
    {"metric": "D08 v2 shape", "value": str(report["d08_shape"])},
    {"metric": "D08 v2 hash", "value": str(report["d08_hash"])},
    {"metric": "high confidence match rate", "value": f'{report["metrics"]["high_conf_match_rate"]:.2%}'},
    {"metric": "major high/medium mapping rate", "value": f'{report["metrics"]["major_high_medium_rate"]:.2%}'},
    {"metric": "final_status", "value": report["final_status"]},
    {"metric": "p4_ready_candidate", "value": str(manifest["p4_ready_candidate"])},
])
display(summary)
display(qa)


,metric,value
0,D01 v2 shape,"[34969, 186]"
1,D01 v2 hash,2f187b5af44c828d4107af12368029caf6b83b6254af75...
2,D03 v2 shape,"[10242, 108]"
3,D03 v2 hash,c6fd569052684502e5bab5758510d3cd945f68ddeaa47f...
4,D08 v2 shape,"[10242, 151]"
5,D08 v2 hash,598b68b31b5358dfd23839d4c138cc64838d05876b7791...
6,high confidence match rate,83.59%
7,major high/medium mapping rate,98.60%
8,final_status,READY_FOR_REAUDIT
9,p4_ready_candidate,True


,check,actual,expected,status
0,D01 canonical grain duplicate,0,0,PASS
1,D03 row count,10242,10242,PASS
2,structure high-confidence match rate,0.835872,>=0.70,PASS
3,major high/medium mapping rate,0.986038,>=0.97,PASS
4,major ambiguous+unknown rate,0.013962,<=0.03,PASS
5,candidate_count>=2 auto-confirmed,0,0,PASS
6,campus conflict auto-confirmed,0,0,PASS
7,Local2 D07 handoff ready,True,True,PASS
8,D08 row count,10242,10242,PASS
9,D07->D08 exact mismatch,0,0,PASS
